# 🚕 Urban Mobility Intelligence

### Understanding When and Where NYC Taxi Demand Becomes Difficult to Serve

## Business Objective

Urban transportation demand is not evenly distributed throughout the day or across the city.

This project uses NYC Yellow Taxi trip data to identify:

- when taxi demand is highest,
- where demand is concentrated,
- and which high-demand periods and locations may experience greater operational pressure.

The goal is to turn trip-level data into insights that could support better transportation planning and resource allocation.

## 1. Data Loading

The analysis begins with the January 2025 NYC Yellow Taxi trip dataset.

January is being used as a development month to establish and validate the analytical methodology before applying the same process to the full year.

In [5]:
import pandas as pd
import numpy as np
df = pd.read_parquet("yellow_tripdata_2025-01.parquet")

print(f"Dataset contains {df.shape[0]:,} trips and {df.shape[1]} columns.")

Dataset contains 3,475,226 trips and 20 columns.


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

## 2. Initial Data Assessment

Before cleaning, the dataset was reviewed for missing values and fields relevant to the project's business questions.

The analysis primarily depends on trip timestamps, pickup/drop-off locations, distance, fare information, and trip duration.

In [7]:
missing = (
    df.isna()
      .sum()
      .loc[lambda x: x > 0]
      .sort_values(ascending=False)
)

missing

passenger_count         540149
RatecodeID              540149
store_and_fwd_flag      540149
congestion_surcharge    540149
Airport_fee             540149
dtype: int64

### Data Quality Decision

Several fields contain missing values, particularly passenger count and certain fare-related attributes. These fields are not required for the core demand analysis, so the missing values will be retained rather than imputed.

## 3. Data Cleaning

The objective of cleaning is to remove records that could distort trip-level mobility analysis while avoiding unnecessary transformations.

The following records will be excluded:

- Trips with zero or negative duration
- Trips exceeding 180 minutes
- Trips with trip distance above 100 miles
- Records with negative fare amounts

Missing values in fields not required for the core analysis will be retained rather than artificially imputed.

In [8]:
df_clean = df.copy()

# Calculate trip duration
df_clean["trip_duration_minutes"] = (
    df_clean["tpep_dropoff_datetime"]
    - df_clean["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Remove records unsuitable for trip-level mobility analysis
df_clean = df_clean[
    (df_clean["trip_duration_minutes"] > 0) &
    (df_clean["trip_duration_minutes"] <= 180) &
    (df_clean["trip_distance"] <= 100) &
    (df_clean["fare_amount"] >= 0)
].copy()

print(f"Original records : {len(df):,}")
print(f"Cleaned records  : {len(df_clean):,}")
print(f"Records removed  : {len(df) - len(df_clean):,}")
print(f"Retention rate   : {len(df_clean) / len(df):.2%}")

Original records : 3,475,226
Cleaned records  : 3,327,575
Records removed  : 147,651
Retention rate   : 95.75%


## 4. Feature Engineering

To support the demand and operational analysis, we derive a small set of time-based features from the pickup timestamp.

These features allow us to compare demand across hours, days of the week, and weekdays vs weekends.

In [9]:
df_clean["pickup_date"] = df_clean["tpep_pickup_datetime"].dt.date
df_clean["pickup_hour"] = df_clean["tpep_pickup_datetime"].dt.hour
df_clean["day_name"] = df_clean["tpep_pickup_datetime"].dt.day_name()

df_clean["day_type"] = np.where(
    df_clean["day_name"].isin(["Saturday", "Sunday"]),
    "Weekend","Weekday")

## 5. Demand Analysis — When Does Demand Peak?

Understanding when taxi demand is highest helps identify periods where transportation capacity may experience greater pressure.

We first examine pickup demand by hour to identify the overall daily demand pattern.

In [13]:
hourly_demand = (df_clean["pickup_hour"].value_counts().sort_index())

hourly_demand

pickup_hour
0      87749
1      60502
2      41003
3      26247
4      18506
5      21013
6      47313
7      97842
8     135260
9     138730
10    144559
11    156025
12    170452
13    180588
14    196137
15    206932
16    209654
17    242013
18    253746
19    210604
20    186277
21    195875
22    172050
23    128498
Name: count, dtype: int64

### Finding

Taxi demand in January 2025 increases substantially through the afternoon and reaches its highest point at **6 PM, with 253,746 pickups**. Demand then declines through the evening.

This indicates a clear late-afternoon/evening demand peak that warrants further investigation by day of week and geographic location.

### Weekday vs Weekend Demand

The overall hourly pattern may hide differences between weekdays and weekends. Comparing the two groups helps determine whether peak demand is concentrated around typical working-day periods or follows a different weekend pattern.

In [16]:
weekday_weekend_demand = (
    df_clean.groupby(["day_type", "pickup_hour"])
    .size()
    .unstack(fill_value=0))

weekday_weekend_demand

pickup_hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
day_type,,,,,,,,,,,,,,,,,,,,,
Weekday,40509,23426,15062,10272,9928,17021,40913,88182,119994,114507,...,144730,155114,156149,188818,200440,161953,145652,154148,129322,87486
Weekend,47240,37076,25941,15975,8578,3992,6400,9660,15266,24223,...,51407,51818,53505,53195,53306,48651,40625,41727,42728,41012


### Finding

Weekday and weekend taxi demand follow different hourly patterns. Weekday demand builds through the afternoon and peaks at **6 PM**, while weekend demand reaches its highest level earlier, around **4 PM**.

This suggests that transportation capacity planning should account for different peak periods on weekdays and weekends rather than using a single peak-hour assumption.

## 6. Geographic Demand Analysis

Pickup location IDs in the trip records correspond to NYC Taxi Zones. We join the official Taxi Zone lookup to translate these IDs into geographic areas and identify where taxi demand is concentrated.

In [30]:
taxi_zones = pd.read_csv("taxi_zone_lookup.csv")

taxi_zones = taxi_zones[["Location ID", "Borough", "Zone"]]

taxi_zones.head()

,Location ID,Borough,Zone
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights


In [21]:
df_clean = df_clean.merge(
    taxi_zones,
    left_on="PULocationID",
    right_on="Location ID",
    how="left"
).drop(columns="Location ID")

### Pickup Demand by Zone

We rank taxi zones by the number of pickups originating from each zone. This identifies the areas with the highest concentration of taxi demand and provides a basis for investigating potential operational pressure.

In [24]:
zone_demand = (
    df_clean.groupby(["Borough", "Zone"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

zone_demand

Borough    Zone                        
Manhattan  Midtown Center                  163997
           Upper East Side South           159709
           Upper East Side North           151573
Queens     JFK Airport                     137340
Manhattan  Times Sq/Theatre District       120028
           Penn Station/Madison Sq West    115375
           Midtown East                    114385
           Lincoln Square East             107230
           Upper West Side South            93253
           Midtown North                    92808
dtype: int64

### Finding

Taxi demand is highly concentrated in a small number of pickup zones. **Midtown Center** recorded the highest demand with **163,997 pickups**, followed by **Upper East Side South** and **Upper East Side North**.

Nine of the top ten pickup zones are in Manhattan, while **JFK Airport** is the only Queens zone in the top ten. This concentration suggests that operational capacity may need to be prioritized in specific high-demand areas rather than distributed evenly across the city.

## 7. Operational Pressure

High demand alone does not necessarily indicate an operational challenge. To identify periods of potential pressure, we compare demand with trip duration and average travel speed.

Longer trips and lower speeds during high-demand periods may indicate that the transportation network is becoming more difficult to serve efficiently.

In [28]:
df_clean["avg_speed_mph"] = (
    df_clean["trip_distance"] /
    (df_clean["trip_duration_minutes"] / 60)
)

df_clean.loc[
    ~df_clean["avg_speed_mph"].between(0, 80),
    "avg_speed_mph"
] = np.nan

In [29]:
hourly_pressure = (
    df_clean.groupby("pickup_hour")
    .agg(
        demand=("pickup_hour", "size"),
        avg_duration_minutes=("trip_duration_minutes", "mean"),
        avg_speed_mph=("avg_speed_mph", "mean")
    )
    .sort_values("demand", ascending=False))

hourly_pressure

,demand,avg_duration_minutes,avg_speed_mph
pickup_hour,,,
18,253756,14.127719,9.367851
17,242035,15.370414,9.219483
19,210617,13.569187,10.671045
16,209674,16.268335,9.739485
15,206962,16.285918,9.795281
14,196169,16.006341,9.966669
21,195891,13.403230,12.518724
20,186292,13.350385,12.263017
13,180616,15.231795,10.122118


### Finding

Peak taxi demand coincides with slower average travel speeds. At **6 PM**, the highest-demand hour, taxis averaged **9.37 mph**, compared with **18.58 mph at 5 AM**, when demand was much lower.

This suggests that peak-demand periods may experience greater operational pressure. However, this analysis shows an association rather than proving that higher demand directly causes slower travel, since traffic and trip characteristics may also influence speed.

## 8. Key Findings & Business Recommendations

### Key Findings

**1. Demand peaks during the evening.**  
January 2025 taxi demand was highest at **6 PM**, with 253,756 pickups. Weekday demand showed a stronger evening peak, while weekend demand peaked earlier in the afternoon.

**2. Demand is concentrated geographically.**  
**Midtown Center** recorded the highest number of pickups with 163,997 trips. Nine of the ten highest-demand pickup zones were located in Manhattan, with JFK Airport being the only Queens zone in the top ten.

**3. Peak demand coincides with slower travel.**  
At 6 PM, taxis averaged **9.37 mph**, compared with **18.58 mph at 5 AM**. This suggests greater operational pressure during periods of high demand, although the relationship does not establish causation.

### Business Recommendations

- Prioritize vehicle availability around major demand hotspots during peak periods.
- Use different capacity-planning strategies for weekdays and weekends because their demand patterns differ.
- Focus operational monitoring on high-demand areas where demand coincides with slower travel speeds.
- Use the full-year analysis to determine whether these patterns are consistent throughout 2025 before making broader operational decisions.